# Train the 6 dropped-from-grid configurations to the full seed count

The submitted Figure 3(d) uses 15 configurations; 6 of them were dropped when
the 33-grid was revised, so the 50-seed rerun skipped them. This notebook
brings exactly those 6 up to the same seed list (parsed from
`item1_all33_seeds.ipynb`), writing into the same `item1_all33/` folder with
the same runner, steps and logging — so `fig3d_15configs.ipynb` picks them up
with no changes. Restart-safe: existing checkpoints are skipped.


In [1]:
import ast, json, re, sys, time
from pathlib import Path

WORKDIR = Path('.').resolve()
sys.path.insert(0, str(WORKDIR / 'shared'))
OUT = WORKDIR / 'item1_all33'; OUT.mkdir(exist_ok=True)

_nb = json.loads((WORKDIR / 'item1_all33_seeds.ipynb').read_text())
_src = next(''.join(c['source']) for c in _nb['cells'] if 'SEEDS = [' in ''.join(c['source']))
_lit = _src[_src.index('SEEDS = ['):]
SEEDS = ast.literal_eval(_lit[_lit.index('['):_lit.index(']') + 1])
N_STEPS, LOG_EVERY, N = 8000, 20, 15000

# the 6 configurations of the submitted figure that the revised grid dropped
DROPPED6 = [  # (gid, teacher, p, r_t, r_s, n, eta)
    ('X1', 'relu', 100, 16, 50, N, 2.0),
    ('X2', 'gelu', 100,  4, 50, N, 0.7),
    ('X3', 'gelu', 100,  8, 50, N, 0.7),
    ('X4', 'gelu', 100, 16, 50, N, 0.7),
    ('X5', 'silu', 100,  4, 50, N, 0.85),
    ('X6', 'silu', 100, 16, 50, N, 0.85),
]
print(f'{len(SEEDS)} seeds x {len(DROPPED6)} configs = {len(SEEDS)*len(DROPPED6)} runs max')


50 seeds x 6 configs = 300 runs max


In [2]:
# build the TODO list: only (config, seed) pairs without a finished checkpoint
import numpy as np
RUNS = []
for gid, ta, p, rt, rs, n, eta in DROPPED6:
    have = 0
    for seed in SEEDS:
        ck = OUT / f'{ta}_p{p}_rt{rt}_rs{rs}_n{n}_eta{eta}_seed{seed}.npz'
        if ck.exists():
            try:
                if int(np.load(ck, allow_pickle=True)['next_t']) >= N_STEPS:
                    have += 1; continue
            except Exception:
                pass    # unreadable/partial -> re-run (runner resumes it)
        RUNS.append((gid, ta, p, rt, rs, n, eta, seed))
    print(f'{gid} {ta} rt={rt:>2} eta={eta:<5} complete={have:>2}/{len(SEEDS)}')
print(f'\n{len(RUNS)} runs to do')


X1 relu rt=16 eta=2.0   complete=50/50
X2 gelu rt= 4 eta=0.7   complete=50/50
X3 gelu rt= 8 eta=0.7   complete=50/50
X4 gelu rt=16 eta=0.7   complete=50/50
X5 silu rt= 4 eta=0.85  complete=50/50
X6 silu rt=16 eta=0.85  complete=50/50

0 runs to do


In [3]:
# multi-machine sharding, same convention as item1_all33_seeds.ipynb
N_PROC      = 4      # worker processes on this machine
WORKER_ID   = 0      # 0 .. NUM_WORKERS-1, unique per machine
NUM_WORKERS = 1
MY_RUNS = [r for i, r in enumerate(RUNS) if i % NUM_WORKERS == WORKER_ID]
print(f'worker {WORKER_ID}/{NUM_WORKERS}: {len(MY_RUNS)} runs')


worker 0/1: 0 runs


In [4]:
import multiprocessing as mp
import concurrent.futures as cf
from parallel_worker import run_full

ARGS = [(str(OUT), N_STEPS, LOG_EVERY, gid, ta, p, rt, rs, n, eta, seed)
        for (gid, ta, p, rt, rs, n, eta, seed) in MY_RUNS]
t0 = time.time(); done = 0
with cf.ProcessPoolExecutor(max_workers=N_PROC,
                            mp_context=mp.get_context('fork')) as ex:
    for gid, seed in ex.map(run_full, ARGS):
        done += 1
        el = time.time() - t0
        rem = el / done * (len(ARGS) - done)
        print(f'[{done}/{len(ARGS)}] {gid} seed={seed} done  '
              f'({el/3600:.2f} h elapsed, ~{rem/3600:.2f} h left)', flush=True)
print(f'finished in {(time.time()-t0)/3600:.2f} h')


finished in 0.00 h


After this finishes, rerun `fig3d_15configs.ipynb` — its seed table should
show 50 everywhere and the fits update automatically.
